## Group access management
- Example:

    - **Data Analysts**: Read-only access to "Gold" data layers, Genie Spaces, and permissions to use SQL warehouses.
    - **Data Engineers**: Permissions to create and manage clusters, Lakeflow pipelines, and write access to "Bronze" and "Silver" data layers.
    - **Machine Learning Engineers**: Entitlements for MLflow experiment tracking, Model Serving, and GPU-enabled cluster policies.
    - **DevOps**: Access to configure service principals, CI/CD pipeline management, and audit log monitoring.
    - **Admins**: Full control over workspace settings, identity management, and Unity Catalog metastores.

### Client with SP permission

In [0]:
from databricks.sdk import WorkspaceClient

# Retrieve SP credentials for API authentication
sp_client_id = dbutils.secrets.get(scope="mgmt-scope", key="sp-id")
sp_secret = dbutils.secrets.get(scope="mgmt-scope", key="sp-secret")

# Initialize client using the Service Principal
w = WorkspaceClient(
    host=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get(),
    client_id=sp_client_id,
    client_secret=sp_secret
)

# Check current permissions for the service principal
sp_me = w.current_user.me()
sp_id = sp_me.id

# List all group entitlements for the SP
sp_groups = [g for g in w.groups.list() if any(m.value == sp_id for m in (g.members or []))]

# Check if SP has group management permissions (simplified check)
has_group_admin = any("admin" in (g.display_name or "").lower() or "manage" in (g.display_name or "").lower() for g in sp_groups)

if not has_group_admin:
    # Grant permissions to create group, create users, and manage a group
    # This requires workspace admin privileges
    # Grant CREATE GROUP, CREATE USER, and MANAGE GROUP to the SP
    perms = [
        "CREATE GROUP",
        "CREATE USER",
        "MANAGE GROUP"
    ]
    for perm in perms:
        sql = f"GRANT {perm} TO `{sp_me.user_name}`"
        spark.sql(sql)

# Granular Access: Add a user or another SP to a specific group
target_group_name = "data-analytics-team"
user_to_add = "analyst@example.com"

# Find group ID and add member
group = next(w.groups.list(filter=f"displayName eq '{target_group_name}'"))
w.groups.patch(
    id=group.id,
    operations=[{"op": "add", "path": "members", "value": [{"display": user_to_add}]}]
)


#### Group

In [0]:
%sql
-- Group 1 Creation and Membership
CREATE GROUP group_example;

#### Creating one user

In [0]:
from databricks.sdk import WorkspaceClient

# Authenticates using environment variables (DATABRICKS_HOST and DATABRICKS_TOKEN)
w = WorkspaceClient()

# Create a new user
new_user = w.users.create(
    display_name="user_one",
    user_name="user_one@example.com"
)

# print(f"Created user with ID: {new_user.id}")

#### Creating multiple users

In [0]:
from databricks.sdk import WorkspaceClient

# Authenticates using environment variables (DATABRICKS_HOST and DATABRICKS_TOKEN)
w = WorkspaceClient()

# Create three new users
users_to_create = [
    {"display_name": "user_four", "user_name": "user_four@example.com"},
    {"display_name": "user_two", "user_name": "user_two@example.com"},
    {"display_name": "user_three", "user_name": "user_three@example.com"}
]

created_users = [w.users.create(**user) for user in users_to_create]

# print([user.id for user in created_users])

#### Adding user to a group

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import iam

# Initializes using environment variables (DATABRICKS_HOST and DATABRICKS_TOKEN)
w = WorkspaceClient()

# 1. Provide the identifiers
group_name = "group_example"
user_email = "user_one@example.com"

# 2. Retrieve IDs for the group and user
# (In production, use w.groups.list(filter=...) or w.users.list(filter=...))
target_group = next(g for g in w.groups.list() if g.display_name == group_name)
target_user = next(u for u in w.users.list() if u.user_name == user_email)

# 3. Add the user to the group using a PATCH operation
w.groups.patch(
    id=target_group.id,
    operations=[
        iam.Patch(
            op=iam.PatchOp.ADD,
            value={
                "members": [
                    {
                        "value": target_user.id
                    }
                ]
            }
        )
    ],
    schemas=[iam.PatchSchema.URN_IETF_PARAMS_SCIM_API_MESSAGES_2_0_PATCH_OP]
)

print(f"Successfully added {user_email} to group {group_name}.")


- getting groups info

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Get all groups and find the one at a specific index (e.g., index 0 for the 1st group)
all_groups = list(w.groups.list())
target_index = 0 

if len(all_groups) > target_index:
    group = all_groups[target_index]
    # print(f"Index {target_index} Group Name: {group.display_name}")
    # print(f"Group ID: {group.id}")

In [0]:
# Grant SELECT permission to group on a table
table_name = "end_to_end_retail.db_landing.tbl_customer_autoloader"
sql = f"GRANT SELECT ON TABLE {table_name} TO `group:{group_name}`"
spark.sql(sql)

### Unity Catalog

#### Granting permission to groups or users in tables with conditions

In [0]:
# Grant SELECT permission to a specific user on a table
table_name = "end_to_end_retail.db_landing.tbl_customer_autoloader"
user_email = "user_one@example.com"
sql = f"GRANT SELECT ON TABLE {table_name} TO `{user_email}`"
spark.sql(sql)

In [0]:
%sql
SHOW GRANTS ON TABLE end_to_end_retail.db_landing.tbl_customer_autoloader

In [0]:
%sql
-- group_example SELECT
GRANT SELECT ON TABLE end_to_end_retail.db_landing.tbl_customer_autoloader TO `group_example`;

-- dataengineers SELECT, MODIFY
GRANT SELECT, MODIFY ON TABLE customers TO `dataengineers`;